# 🚀 Production Pipeline v2.1: WikiMatrix SFT QLoRA + DPO Alignment + Hybrid RAG Benchmark
Notebook huấn luyện 2 giai đoạn (**Supervised Fine-Tuning QLoRA** + **Direct Preference Optimization DPO**) trên dữ liệu **WikiMatrix (`en-vi`)** và chấm điểm đa chiều (**SacreBLEU**, **chrF++**, **Latency**).

**Trước khi chạy:** chọn *Runtime → Change runtime type → GPU (T4 trở lên)*, tải project dạng ZIP lên ở Bước 0, rồi chạy lần lượt từ trên xuống. Cần Internet để tải WikiMatrix/model và xác nhận quyền truy cập Google Drive; adapters sẽ được lưu vào Drive.

## Bước 0: Đưa mã nguồn lên Colab
Tải file `wikimatrix-rag-nmt-v2.zip` lên khi được hỏi (hoặc `git clone` repo của bạn rồi `os.chdir` vào thư mục đó).

In [ ]:
import os
import zipfile

if not os.path.exists("src"):
    from google.colab import files

    uploaded = files.upload()  # chọn file zip của dự án
    zip_name = next(iter(uploaded))
    zipfile.ZipFile(zip_name).extractall(".")
    project_dir = next(d for d in os.listdir(".") if os.path.isdir(d) and os.path.exists(os.path.join(d, "src")))
    os.chdir(project_dir)
print("Thư mục làm việc:", os.getcwd())

In [ ]:
# Cài dependency đã pin, nhưng giữ PyTorch GPU có sẵn trong Colab.
from pathlib import Path
import subprocess
import sys

requirements = [
    line for line in Path("requirements.txt").read_text().splitlines()
    if not line.strip().startswith("torch==")
]
requirements_path = Path("/tmp/requirements-colab.txt")
requirements_path.write_text("\n".join(requirements) + "\n")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])

In [ ]:
# Colab runtimes are temporary, so save adapters in Drive and verify the GPU before training.
import os

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
drive.mount("/content/drive")

from src.core.settings import load_settings

settings = load_settings()
if settings.device != "cuda":
    raise RuntimeError(f"Ứng dụng không nhận GPU (device={settings.device!r}).")

model_root = "/content/drive/MyDrive/wikimatrix-qwen-models"
os.makedirs(model_root, exist_ok=True)
settings.model.lora_output_dir = os.path.join(model_root, "sft-adapter")
settings.model.dpo_output_dir = os.path.join(model_root, "dpo-adapter")

# Small per-device batches fit a free Colab T4; accumulation preserves effective batch size.
settings.qlora.train_batch_size = 1
settings.qlora.eval_batch_size = 1
settings.qlora.gradient_accumulation_steps = 16
settings.dpo.batch_size = 1
print("Thiết bị:", settings.device, "| adapters lưu tại:", model_root)

## Bước 1: Huấn luyện SFT 4-bit QLoRA (loss chỉ tính trên bản dịch tiếng Việt)

In [ ]:
from src.training.sft_qlora import run_sft_qlora

run_sft_qlora(settings=settings)

## Bước 2: Xem thử dữ liệu Preference Triplets cho DPO

In [ ]:
from src.training.dpo_align import build_synthetic_dpo_dataset

dpo_ds = build_synthetic_dpo_dataset(settings=settings, num_samples=2000)
print("Sample DPO Triplet:", dpo_ds[0])

## Bước 3: Huấn luyện DPO (tiếp tục từ adapter SFT; reference model = bản SFT)

In [ ]:
from src.training.dpo_align import run_dpo_training

run_dpo_training(num_samples=2000, settings=settings)

## Bước 4: Chạy Benchmark đa chiều (SacreBLEU + chrF++ + Latency ms)

In [ ]:
from src.evaluation.benchmark import evaluate_system

evaluate_system(num_samples=100, settings=settings)

## Bước 5: Đóng gói các Adapter để tải về máy cá nhân

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = "/content/wikimatrix-qwen-adapters.zip"
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for adapter_name, adapter_dir in (("sft-adapter", settings.model.lora_output_dir), ("dpo-adapter", settings.model.dpo_output_dir)):
        for file_path in Path(adapter_dir).rglob("*"):
            if file_path.is_file():
                archive.write(file_path, arcname=f"{adapter_name}/{file_path.relative_to(adapter_dir)}")
from google.colab import files

files.download(archive_path)